# 2.2 **Build** a Regularized Logistic Regression Model - Predict Student Departure with L1, L2, and ElasticNet

## Model Cycle: The 5 Key Steps

### **1. Build the Model : Create the pipeline with regularization.**  
### 2. Train the Model : Fit the model on the training data.  
### 3. Generate Predictions : Use the trained model to make predictions.  
### 4. Evaluate the Model : Assess performance using evaluation metrics.  
### 5. Improve the Model : Tune hyperparameters for optimal performance.

## Introduction

In this notebook, we build upon the baseline logistic regression model from Course 2 by adding regularization. We will create three regularized variants:

1. **L2 (Ridge)**: Shrinks all coefficients, handles multicollinearity
2. **L1 (Lasso)**: Performs feature selection by zeroing coefficients
3. **ElasticNet**: Combines L1 and L2 benefits

We use the same preprocessing pipeline and data from Course 2, allowing direct comparison between models.

### Learning Objectives

By the end of this notebook, you will be able to:

1. Build regularized logistic regression pipelines in scikit-learn
2. Understand the key hyperparameters (`C`, `penalty`, `l1_ratio`)
3. Create multiple model variants for comparison

## 1. Load Dependencies and Data

In [1]:
# Data is in the project's data/ directory
# No Google Drive mount needed when running locally

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import pickle

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder

pd.options.display.max_columns = None

In [3]:
# Set up file paths - using Course 2 data
project_path = '/content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3'
data_filepath = '/data/'
models_filepath = '/models/'

In [4]:
# Load training data
df_training = pd.read_csv(f'{project_path}{data_filepath}training.csv')

print(f"Training data shape: {df_training.shape}")
print(f"\nTarget distribution:")
print(df_training['SEM_3_STATUS'].value_counts(normalize=True))

Training data shape: (19844, 27)

Target distribution:
SEM_3_STATUS
E    0.867113
N    0.132887
Name: proportion, dtype: float64


In [5]:
# Define feature matrix and target
X_train = df_training
y_train = df_training['SEM_3_STATUS']

## 2. Quick Recap: The Baseline Model

In Course 2, we built a baseline logistic regression model without regularization (`penalty=None`). Here's a quick recap of our preprocessing setup:

In [6]:
# Feature groupings from Course 2

# Columns with bounded values (e.g., 0–4 GPA or 0–1 ratios)
minmax_columns = [
    'HS_GPA',
    'GPA_1', 'GPA_2',
    'DFW_RATE_1', 'DFW_RATE_2'
]

# Columns with larger, more variable values
standard_columns = [
    'UNITS_ATTEMPTED_1', 'UNITS_ATTEMPTED_2',
]

# Categorical columns for one-hot encoding
categorical_columns = [
    'GENDER',
    'RACE_ETHNICITY',
    'FIRST_GEN_STATUS',
]

In [7]:
# Preprocessing transformer (same as Course 2)
preprocessor = ColumnTransformer(
    transformers=[
        ('minmax', MinMaxScaler(), minmax_columns),
        ('standard', StandardScaler(), standard_columns),
        ('onehot', OneHotEncoder(handle_unknown='ignore',
                                  drop=['Female', 'Other', 'Unknown'],
                                  sparse_output=False), categorical_columns)
    ],
    remainder='drop'
)

print("Preprocessor configured successfully.")

Preprocessor configured successfully.


## 3. Building Regularized Models

Now we create three regularized versions of our logistic regression model. Each uses the same preprocessing but with different regularization settings.

### Key Hyperparameters

| Parameter | Description | Values |
|:----------|:------------|:-------|
| `penalty` | Type of regularization | 'l1', 'l2', 'elasticnet', None |
| `C` | Inverse of regularization strength | float > 0 (default=1.0) |
| `solver` | Optimization algorithm | 'saga' (required for L1/ElasticNet) |
| `l1_ratio` | ElasticNet mixing (1=L1, 0=L2) | float between 0 and 1 |

### 3.1 L2 Regularization (Ridge)

L2 regularization is scikit-learn's default. It shrinks all coefficients proportionally but doesn't zero any out.

In [8]:
model_l2 = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(penalty='l2',
                                      C=1.0,
                                      random_state=42,
                                      solver='liblinear',
                                      class_weight='balanced'
                                      ))
])

print("L2 Regularized Logistic Regression Model:")
print(model_l2)

from sklearn import set_config
set_config(display='diagram')
model_l2

L2 Regularized Logistic Regression Model:
Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                                  ['HS_GPA', 'GPA_1', 'GPA_2',
                                                   'DFW_RATE_1',
                                                   'DFW_RATE_2']),
                                                 ('standard', StandardScaler(),
                                                  ['UNITS_ATTEMPTED_1',
                                                   'UNITS_ATTEMPTED_2']),
                                                 ('onehot',
                                                  OneHotEncoder(drop=['Female',
                                                                      'Other',
                                                                      'Unknown'],
                                                                handle_unknown='ignore',
                     

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                                  ['HS_GPA', 'GPA_1', 'GPA_2',
                                                   'DFW_RATE_1',
                                                   'DFW_RATE_2']),
                                                 ('standard', StandardScaler(),
                                                  ['UNITS_ATTEMPTED_1',
                                                   'UNITS_ATTEMPTED_2']),
                                                 ('onehot',
                                                  OneHotEncoder(drop=['Female',
                                                                      'Other',
                                                                      'Unknown'],
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['GENDER', 'RACE_ETHNICITY',
                                                   'FIRST_GEN_STATUS'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', random_state=42,
                                    solver='liblinear'))])

### 3.2 L1 Regularization (Lasso)

L1 regularization can shrink coefficients to exactly zero, effectively performing feature selection. Note that L1 requires the 'saga' solver.

In [9]:
model_l1 = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(penalty='l1',
                                      C=1.0,
                                      random_state=42,
                                      solver='saga', # 'saga' solver is required for L1 penalty
                                      class_weight='balanced'
                                     ))
])

print("L1 Regularized (Lasso) Logistic Regression Model:")
print(model_l1)

from sklearn import set_config
set_config(display='diagram')
model_l1

L1 Regularized (Lasso) Logistic Regression Model:
Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                                  ['HS_GPA', 'GPA_1', 'GPA_2',
                                                   'DFW_RATE_1',
                                                   'DFW_RATE_2']),
                                                 ('standard', StandardScaler(),
                                                  ['UNITS_ATTEMPTED_1',
                                                   'UNITS_ATTEMPTED_2']),
                                                 ('onehot',
                                                  OneHotEncoder(drop=['Female',
                                                                      'Other',
                                                                      'Unknown'],
                                                                handle_unknown='ignore',
             

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                                  ['HS_GPA', 'GPA_1', 'GPA_2',
                                                   'DFW_RATE_1',
                                                   'DFW_RATE_2']),
                                                 ('standard', StandardScaler(),
                                                  ['UNITS_ATTEMPTED_1',
                                                   'UNITS_ATTEMPTED_2']),
                                                 ('onehot',
                                                  OneHotEncoder(drop=['Female',
                                                                      'Other',
                                                                      'Unknown'],
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['GENDER', 'RACE_ETHNICITY',
                                                   'FIRST_GEN_STATUS'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', penalty='l1',
                                    random_state=42, solver='saga'))])

### 3.3 ElasticNet Regularization

ElasticNet combines L1 and L2 penalties. The `l1_ratio` parameter controls the mix:
- `l1_ratio=1.0`: Pure L1 (Lasso)
- `l1_ratio=0.0`: Pure L2 (Ridge)  
- `l1_ratio=0.5`: Equal mix of both

In [10]:
model_elasticnet = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(penalty='elasticnet',
                                      C=1.0,
                                      random_state=42,
                                      solver='saga', # 'saga' solver is required for ElasticNet penalty
                                      l1_ratio=0.5, # 0.5 for an equal mix of L1 and L2
                                      class_weight='balanced'
                                     ))
])

print("ElasticNet Regularized Logistic Regression Model:")
print(model_elasticnet)

from sklearn import set_config
set_config(display='diagram')
model_elasticnet

ElasticNet Regularized Logistic Regression Model:
Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                                  ['HS_GPA', 'GPA_1', 'GPA_2',
                                                   'DFW_RATE_1',
                                                   'DFW_RATE_2']),
                                                 ('standard', StandardScaler(),
                                                  ['UNITS_ATTEMPTED_1',
                                                   'UNITS_ATTEMPTED_2']),
                                                 ('onehot',
                                                  OneHotEncoder(drop=['Female',
                                                                      'Other',
                                                                      'Unknown'],
                                                                handle_unknown='ignore',
             

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                                  ['HS_GPA', 'GPA_1', 'GPA_2',
                                                   'DFW_RATE_1',
                                                   'DFW_RATE_2']),
                                                 ('standard', StandardScaler(),
                                                  ['UNITS_ATTEMPTED_1',
                                                   'UNITS_ATTEMPTED_2']),
                                                 ('onehot',
                                                  OneHotEncoder(drop=['Female',
                                                                      'Other',
                                                                      'Unknown'],
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['GENDER', 'RACE_ETHNICITY',
                                                   'FIRST_GEN_STATUS'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', l1_ratio=0.5,
                                    penalty='elasticnet', random_state=42,
                                    solver='saga'))])

### Model Comparison Summary

In [11]:
models_dict = {
    "L2 Ridge": model_l2,
    "L1 Lasso": model_l1,
    "ElasticNet": model_elasticnet
}

print("\n--- Model Hyperparameter Summary ---\n")
for name, model_pipeline in models_dict.items():
    classifier = model_pipeline.named_steps['classifier']
    print(f"Model: {name}")
    print(f"  Penalty: {classifier.penalty}")
    print(f"  C: {classifier.C}")
    print(f"  Solver: {classifier.solver}")
    if hasattr(classifier, 'l1_ratio') and classifier.penalty == 'elasticnet':
        print(f"  L1 Ratio: {classifier.l1_ratio}")
    print("\n")


--- Model Hyperparameter Summary ---

Model: L2 Ridge
  Penalty: l2
  C: 1.0
  Solver: liblinear


Model: L1 Lasso
  Penalty: l1
  C: 1.0
  Solver: saga


Model: ElasticNet
  Penalty: elasticnet
  C: 1.0
  Solver: saga
  L1 Ratio: 0.5




## 4. Save Models for Future Use

We save these untrained model pipelines so they can be loaded and trained in subsequent notebooks.

In [12]:
import os

# Ensure the models directory exists
full_models_path = f'{project_path}{models_filepath}'
os.makedirs(full_models_path, exist_ok=True)

for name, model_pipeline in models_dict.items():
    # Generate a descriptive filename
    filename = name.lower().replace(' ', '_') + '_logistic.pkl'
    filepath = os.path.join(full_models_path, filename)

    # Save the model
    with open(filepath, 'wb') as f:
        pickle.dump(model_pipeline, f)
    print(f"Saved {name} model to {filepath}")

Saved L2 Ridge model to /content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3/models/l2_ridge_logistic.pkl
Saved L1 Lasso model to /content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3/models/l1_lasso_logistic.pkl
Saved ElasticNet model to /content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3/models/elasticnet_logistic.pkl


## 5. Summary

In this notebook, we built three regularized logistic regression models:

| Model | Key Characteristics | Best For |
|:------|:--------------------|:---------|
| **L2 (Ridge)** | Shrinks all coefficients, stable | Many small effects |
| **L1 (Lasso)** | Zeros out coefficients | Feature selection |
| **ElasticNet** | Combines L1 + L2 | Correlated features |

### Key Points

1. All models use `class_weight='balanced'` to handle class imbalance
2. L1 and ElasticNet require `solver='saga'`
3. The `C` parameter is the inverse of regularization strength
4. We're using default `C=1.0`—we'll tune this later

### Next Steps

In the next notebook, we will:
1. Train these models on our data
2. Compare coefficient values across regularization types
3. Examine which features are selected by L1/ElasticNet

**Proceed to:** `2.3 Train and Compare Regularized Logistic Regression Models`